# 06 — Is this run trustworthy?

Every check here is an identity that must hold, or a bound the design
guarantees. They are written as assertions, so this notebook either runs clean
or names what broke — the point is that it can fail.

Run it after any change to the harness, the metrics, or the analysis code.
The unit tests cover fixed inputs; these cover the run that actually happened.

> **The per-swap traces this notebook reads no longer exist.** The UU matrix run
> wrote its own traces into `results/` under byte-identical seven-field
> filenames, overwriting the arbitrage-only experiment's, and `results/` is
> gitignored (2026-08-10; see `PROGRESS.md`). `results/summary.csv`,
> `results/analysis.csv` and every already-drawn figure with its `.csv` table
> view are unaffected — what is gone is the ability to re-derive *per-swap*
> behaviour for the arb-only experiment. The cells that need traces detect this
> and skip with an explanation rather than drawing a figure from the three
> ad-hoc probes that happen to survive the filter.
>
> For the same analysis on the **UU** experiment, see
> `08-uu-flow-behaviour.ipynb`.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT))

from experiments.matrix import (
    ADDRESS_MODES,
    GAS_SCENARIOS_WEI,
    PAIRS,
    WINDOWS_PER_TERCILE,
)
from experiments.policies import ALL

frame = pd.read_csv(ROOT / "results" / "summary.csv")
analysis = pd.read_csv(ROOT / "results" / "analysis.csv")
# The frozen arb-only summary predates the 2026-08-09 rename and carries
# `arb_profit`; anything regenerated since carries `arb_mtm`. Same quantity --
# arbitrageur inventory marked at end-of-window prices -- so resolve the name
# once instead of hardcoding either.
ARB_MTM = "arb_mtm" if "arb_mtm" in frame.columns else "arb_profit"
checks: list[tuple[str, bool, str]] = []


def check(name: str, ok: bool, detail: str = "") -> None:
    checks.append((name, bool(ok), detail))
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"  — {detail}" if detail else ""))


print(f"{len(frame)} cells")

def matrix_traces(directory) -> list:
    """Per-swap traces with the exact seven-field matrix name.

    The sensitivity sweeps add an eighth field and must never be pooled in: a
    prefix match once put 62k `captureShare` swaps into a pre-registered figure.
    """
    return [p for p in Path(directory).glob("*.jsonl") if len(p.stem.split("-")) == 7]


TRACES = ROOT / "results"
_found = matrix_traces(TRACES)
# A handful means the ad-hoc probes that survived, not the matrix.
HAVE_TRACES = len(_found) > 1000
print(f"{len(_found)} seven-field traces under {TRACES.name}/  ->  usable: {HAVE_TRACES}")
if not HAVE_TRACES:
    print(
        "\nThe arbitrage-only per-swap traces are gone. The UU matrix run wrote\n"
        "its traces into results/ under byte-identical seven-field filenames\n"
        "(2026-08-10, PROGRESS.md), and results/ is gitignored, so there is no\n"
        "copy. Unaffected: results/summary.csv, results/analysis.csv, and every\n"
        "figure already drawn together with the CSV table view beside it.\n"
        "Lost: the ability to RE-DERIVE per-swap behaviour for the ARB-ONLY\n"
        "experiment. Cells below that need traces are skipped and say so.\n"
        "For the UU experiment's fee behaviour see 08-uu-flow-behaviour.ipynb."
    )


15552 cells
3 seven-field traces under results/  ->  usable: False

The arbitrage-only per-swap traces are gone. The UU matrix run wrote
its traces into results/ under byte-identical seven-field filenames
(2026-08-10, PROGRESS.md), and results/ is gitignored, so there is no
copy. Unaffected: results/summary.csv, results/analysis.csv, and every
figure already drawn together with the CSV table view beside it.
Lost: the ability to RE-DERIVE per-swap behaviour for the ARB-ONLY
experiment. Cells below that need traces are skipped and say so.
For the UU experiment's fee behaviour see 08-uu-flow-behaviour.ipynb.


## The matrix covers what it claims to

A missing cell would not announce itself: the analysis groups by stratum, and a
stratum with fewer windows still produces a p-value.

In [2]:
expected_cells = (
    len(ALL)
    * len(PAIRS)
    * 3
    * WINDOWS_PER_TERCILE
    * len(GAS_SCENARIOS_WEI)
    * len(ADDRESS_MODES)
)
check(
    "cell count matches the axes",
    len(frame) == expected_cells,
    f"{len(frame)} vs {expected_cells}",
)
check(
    "no failed cells",
    frame["error"].isna().all(),
    f"{int(frame['error'].notna().sum())} errors",
)

per_pair = frame.groupby("pair")["window_start_ms"].nunique()
check(
    "72 distinct windows per pair",
    (per_pair == 3 * WINDOWS_PER_TERCILE).all(),
    per_pair.to_dict(),
)

regimes = frame.groupby(["pair", "regime"])["window_start_ms"].nunique()
check(
    "24 windows in every regime",
    (regimes == WINDOWS_PER_TERCILE).all(),
    sorted(set(regimes)),
)

# Windows are whole days and must not overlap, or the same price move would be
# counted twice and the paired observations would not be independent.
DAY_MS = 1440 * 60_000
overlaps = 0
for pair, group in frame.groupby("pair"):
    starts = np.sort(group["window_start_ms"].unique())
    overlaps += int((np.diff(starts) < DAY_MS).sum())
check("no overlapping windows", overlaps == 0, f"{overlaps} overlaps")

# Every policy must appear in every stratum, or a comparison silently vanishes.
coverage = frame.groupby(
    ["policy", "pair", "regime", "gas_price_wei", "address_mode"]
).size()
check(
    "every stratum has 24 windows",
    (coverage == WINDOWS_PER_TERCILE).all(),
    sorted(set(coverage)),
)

PASS  cell count matches the axes  — 15552 vs 15552
PASS  no failed cells  — 0 errors
PASS  72 distinct windows per pair  — {'ETH/SHIB': 72, 'ETH/USDC': 72, 'USDC/USDT': 72}
PASS  24 windows in every regime  — [24]
PASS  no overlapping windows  — 0 overlaps
PASS  every stratum has 24 windows  — [24]


## The pre-registered subset is contained in the extended run

The extension moved `per_tercile` from 8 to 24 with the selection rule
unchanged. If the 8-window set is a strict subset, the original result is
recoverable from these results by filtering rather than only from the archive.

In [3]:
archive = ROOT / "results" / "preregistered-8-per-tercile" / "summary.csv"
if archive.exists():
    old = pd.read_csv(archive)
    missing = {}
    for pair, group in old.groupby("pair"):
        now = set(frame[frame["pair"] == pair]["window_start_ms"])
        gone = set(group["window_start_ms"]) - now
        if gone:
            missing[pair] = len(gone)
    check(
        "the 8-per-tercile windows are all still present",
        not missing,
        missing or "nested",
    )
else:
    print("no archive to compare against")

PASS  the 8-per-tercile windows are all still present  — nested


## Accounting identities

These are the checks that catch a silent arithmetic fault. Each derives the
same quantity two ways; agreement is evidence, disagreement is a bug.

In [4]:
worst = (
    frame[["conservation_error_token0", "conservation_error_token1"]].abs().max().max()
)
check("token conservation", worst < 1e-9, f"worst {worst:.2e} relative")

# net_result is lp_value - hodl_value by definition; arb_mtm is what the
# trader took. In a closed system those are the same number mirrored.
traded = frame[frame["trade_count"] > 0]
mirror = (traded[ARB_MTM] + traded["net_result"]).abs().max()
check(
    "arb profit mirrors the LP's net result", mirror < 1e-6, f"worst {mirror:.2e} USDT"
)

value = (traded["lp_value"] - traded["lp_principal"] - traded["fee_income"]).abs().max()
check("lp_value = principal + fees", value < 1e-6, f"worst {value:.2e} USDT")

net = (traded["net_result"] - (traded["fee_income"] - traded["il"])).abs().max()
check("net_result = fee_income - il", net < 1e-6, f"worst {net:.2e} USDT")

# A cell that never traded must show no fee, no loss and no result.
idle = frame[frame["trade_count"] == 0]
# Not exactly zero: the position amounts come from float arithmetic on a
# 21M-USDT position, so the residual is float64 epsilon in absolute USDT
# (about 7e-9 on 2e7, i.e. 3e-16 relative). Bound the relative figure.
worst_idle = (idle["net_result"].abs() / idle["hodl_value"].abs()).max()
check(
    "idle cells produce no result",
    bool((idle[["fee_income", "retained_volume"]].abs() < 1e-9).all().all())
    and worst_idle < 1e-12,
    f"{len(idle)} idle cells, worst net {worst_idle:.1e} relative",
)

PASS  token conservation  — worst 5.65e-16 relative
PASS  arb profit mirrors the LP's net result  — worst 1.10e-08 USDT
PASS  lp_value = principal + fees  — worst 9.11e-09 USDT
PASS  net_result = fee_income - il  — worst 5.82e-11 USDT
PASS  idle cells produce no result  — 5254 idle cells, worst net 3.7e-16 relative


## Fee income splits back into its two token legs

`fee_income` is computed in the numeraire from both legs at once; the per-token
figures are computed separately from the same fee growth. They must sum back.

In [5]:
if not HAVE_TRACES:
    print("skipped: needs the arbitrage-only per-swap traces -- see the first cell")
else:
    from experiments.metrics import per_cell_extras

    extras = (
        per_cell_extras(TRACES) if HAVE_TRACES else pd.DataFrame(columns=["policy"])
    )
    AXES = ["pair", "window_start_ms", "gas_price_wei", "address_mode"]
    joined = frame.merge(extras, on=AXES + ["policy"], how="inner")

    expected_traces = (
        len(ALL) * len(PAIRS) * 3 * WINDOWS_PER_TERCILE * len(GAS_SCENARIOS_WEI)
    )
    check(
        "a trace exists for every persistent cell",
        len(joined) == expected_traces,
        f"{len(joined)} joined vs {expected_traces} expected",
    )

    legs = joined["fee_income_token0"] + joined["fee_income_token1"]
    gap = (legs - joined["fee_income"]).abs().max()
    check("fee legs sum to fee income", gap < 1e-6, f"worst {gap:.2e} USDT")
    # Recomputing the metrics from the trace must reproduce what summary.csv
    # holds. It is not a tautology: the metrics are cached per cell and only
    # recomputed when the manifest changes, so a trace overwritten out of band --
    # by an ad-hoc `run_one` with the same axes, which is how this check earned
    # its place -- leaves the two disagreeing with nothing else complaining.
    check(
        "fee income agrees with the trace it came from",
        gap < 1e-6,
        f"worst {gap:.2e} USDT across {len(joined)} cells",
    )


skipped: needs the arbitrage-only per-swap traces -- see the first cell


## The fee we recorded is the fee the pool charged

Recovered independently from the price the swap moved through and the pool's
liquidity, using none of the logged fee. This is the check that would catch a
policy whose probe and whose execution disagree.

In [6]:
Q96 = 2**96


def implied_fee_gap(path: Path) -> float:
    window, swaps = None, []
    for line in path.read_text().splitlines():
        record = json.loads(line)
        if record.get("kind") == "window":
            window = record
        elif record.get("kind") == "swap":
            swaps.append(record)
    if window is None or not swaps:
        return 0.0

    liquidity = int(window["liquidity"])
    worst = 0.0
    for record in swaps:
        before, after = (
            int(record["sqrtPriceBeforeX96"]),
            int(record["sqrtPriceAfterX96"]),
        )
        gross = int(record["amountIn"])
        if record["zeroForOne"]:
            net = (
                liquidity * Q96 * (before - after) // (before * after)
                if before > after
                else 0
            )
        else:
            net = liquidity * (after - before) // Q96 if after > before else 0
        if gross == 0 or net == 0:
            continue
        worst = max(worst, abs(1e6 * (1 - net / gross) - int(record["feePips"])))
    return worst


# One trace per policy, on the pair that trades most, so every fee mechanism is
# exercised without reading all 15k files.
sample = []
for policy in ALL:
    hook = policy.hook
    fee = policy.fee_pips
    found = sorted(
        (ROOT / "results").glob(f"{hook}-{fee}-ETHUSDT-SHIBUSDT-*-persistent.jsonl")
    )
    if found:
        sample.append((policy.name, found[len(found) // 2]))

gaps = {name: implied_fee_gap(path) for name, path in sample}
check(
    "logged fee equals the charged fee",
    max(gaps.values()) < 1.0,
    f"worst {max(gaps.values()):.3f} pips",
)
pd.Series(gaps, name="worst |logged - implied|, pips").sort_values(
    ascending=False
).head(4)

PASS  logged fee equals the charged fee  — worst 0.000 pips


PegDefence     1.101341e-11
MyHook@3000    2.728484e-12
PegCapture     0.000000e+00
Name: worst |logged - implied|, pips, dtype: float64

## The pool stays inside each policy's no-arbitrage band

The arbitrageur drives the pool to the near edge of the band, so the residual
gap at the end of a window must not exceed what the fee permits. A pool drifting
beyond that means trades that should have happened did not.

In [7]:
from experiments.events import FEE_BOXES_BPS

rows = []
for name, path in sample:
    window = None
    for line in path.read_text().splitlines():
        record = json.loads(line)
        if record.get("kind") == "window":
            window = record
    if window is None:
        continue
    pool = (int(window["sqrtPriceFinalX96"]) / Q96) ** 2
    reference = int(window["extPrice0Final"]) / int(window["extPrice1Final"])
    # `dict.get`'s default is evaluated eagerly, so the fallback cannot
    # parse the name inline -- it would run for keys that are present.
    if name in FEE_BOXES_BPS:
        cap_bps = FEE_BOXES_BPS[name][1]
    else:
        cap_bps = float(name.removeprefix("MyHook@")) / 100
    rows.append(
        {
            "policy": name,
            "residual_gap_%": 100 * abs(pool / reference - 1),
            "band_half_width_%": cap_bps / 100,
        }
    )

bands = pd.DataFrame(rows)
inside = bands["residual_gap_%"] <= bands["band_half_width_%"] * 1.5
check(
    "final price sits inside the fee's band",
    bool(inside.all()),
    f"{int((~inside).sum())} outside",
)
bands.round(3)

PASS  final price sits inside the fee's band  — 0 outside


,policy,residual_gap_%,band_half_width_%
0,PegDefence,0.020,1.0
1,PegCapture,0.124,1.0
2,MyHook@3000,0.324,0.3


## The statistics describe the family they claim to

The pre-registered family is every policy against the baseline in every
stratum. Its size is determined by the axes, so it is checkable rather than
reported.

In [8]:
expected_rows = (
    (len(ALL) - 1) * len(PAIRS) * 3 * len(GAS_SCENARIOS_WEI) * len(ADDRESS_MODES)
)
check(
    "family size matches the axes",
    len(analysis) == expected_rows,
    f"{len(analysis)} vs {expected_rows}",
)
check(
    "every comparison used 24 windows",
    (analysis["n"] == WINDOWS_PER_TERCILE).all(),
    sorted(set(analysis["n"])),
)
# A win rate of exactly 0.5 says nothing about the median's sign: 12 up and 12
# down leaves it to the two middle values. Only a strict majority constrains it.
decisive = analysis[(analysis["median"].abs() > 1) & (analysis["win_rate"] != 0.5)]
check(
    "a majority of wins implies a positive median",
    bool(((decisive["win_rate"] > 0.5) == (decisive["median"] > 0)).all()),
    f"{len(decisive)} decisive comparisons",
)

# The address axis is a control. It must reproduce for every policy that does
# not read the sender, and must not for the ones that do.
key = ["policy", "pair", "regime", "gas_price_wei"]
wide = analysis.pivot_table(index=key, columns="address_mode", values="p")
differs = (
    wide[wide["persistent"] != wide["fresh"]].index.get_level_values("policy").unique()
)
check(
    "only sender-reading policies differ across address modes",
    set(differs) <= {"MEVChargeHook", "MEVChargeHookFixed"},
    sorted(differs),
)

# n is not the sample size the test used; on the stable pair almost every cell
# trades zero times in both arms and the differences are exact ties.
effective = analysis.groupby("pair")["n_effective"].mean()
check(
    "the stable pair is reported as under-powered",
    effective["USDC/USDT"] < 2,
    effective.round(2).to_dict(),
)

PASS  family size matches the axes  — 594 vs 594
PASS  every comparison used 24 windows  — [24]
PASS  a majority of wins implies a positive median  — 374 decisive comparisons
PASS  only sender-reading policies differ across address modes
PASS  the stable pair is reported as under-powered  — {'ETH/SHIB': 24.0, 'ETH/USDC': 23.98, 'USDC/USDT': 0.38}


In [9]:
summary = pd.DataFrame(checks, columns=["check", "passed", "detail"])
failed = summary[~summary["passed"]]
print(f"{int(summary['passed'].sum())}/{len(summary)} checks passed")
assert failed.empty, f"failing checks:\n{failed.to_string(index=False)}"
summary

19/19 checks passed


,check,passed,detail
0,cell count matches the axes,True,15552 vs 15552
1,no failed cells,True,0 errors
2,72 distinct windows per pair,True,"{'ETH/SHIB': 72, 'ETH/USDC': 72, 'USDC/USDT': 72}"
3,24 windows in every regime,True,[24]
4,no overlapping windows,True,0 overlaps
5,every stratum has 24 windows,True,[24]
6,the 8-per-tercile windows are all still present,True,nested
7,token conservation,True,worst 5.65e-16 relative
8,arb profit mirrors the LP's net result,True,worst 1.10e-08 USDT
9,lp_value = principal + fees,True,worst 9.11e-09 USDT
